In [1]:
!git clone https://github.com/vuog23/vlm-geo.git

Cloning into 'vlm-geo'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 43 (delta 7), reused 42 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 31.68 KiB | 6.33 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [2]:
import sys
sys.path.append("/kaggle/working/vlm-geo")

In [3]:
from src.evaluate.imagenet_eval import ImageNetEval
from src.utils.geode_imagenet_mapping import GEODE_TO_IMAGENET_IDX

In [ ]:
GEODE_TEST_ROOT = r"/kaggle/input/datasets/trieuvuongnguyen/geode/GeoDE_split/test"
MODEL_CONFIGS = {
    "convnextv2_base": "convnextv2_base.fcmae_ft_in22k_in1k",
    "swinv2_base": "swinv2_base_window12to24_192to384.ms_in22k_ft_in1k",
    "deit3_base": "deit3_base_patch16_224.fb_in22k_ft_in1k",
    "dinov2_vitb14": "vit_base_patch14_dinov2.lvd142m",
}

START_BATCH_SIZE = {
    "convnextv2_base": 64,
    "swinv2_base": 4,
    "deit3_base": 64,
    "dinov2_vitb14": 32,
}

In [5]:
import gc
import torch
import pickle

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def evaluate_model_safely(
    short_name,
    model_name,
    root_path,
    target_to_imagenet_idx,
    region_index=2,
    start_batch_size=64,
    min_batch_size=1,
):
    batch_size = start_batch_size

    while batch_size >= min_batch_size:
        clear_gpu()

        print()
        print("=" * 80)
        print("Evaluating:", short_name)
        print("timm model:", model_name)
        print("batch size:", batch_size)
        print("=" * 80)
        print()

        evaluator = None

        try:
            evaluator = ImageNetEval(
                root_path=root_path,
                model_name=model_name,
                target_to_imagenet_idx=target_to_imagenet_idx,
                batch_size=batch_size,
                region_index=region_index,
            )

            results = evaluator.evaluate()
            evaluator.print_summary(results)

            del evaluator
            clear_gpu()

            return results

        except torch.cuda.OutOfMemoryError:
            print(f"\nCUDA OOM with batch_size={batch_size}. Retrying smaller...\n")

            if evaluator is not None:
                del evaluator

            clear_gpu()
            batch_size = batch_size // 2

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"\nCUDA OOM with batch_size={batch_size}. Retrying smaller...\n")

                if evaluator is not None:
                    del evaluator

                clear_gpu()
                batch_size = batch_size // 2
            else:
                raise e

    raise RuntimeError(f"{short_name} failed even with batch_size={min_batch_size}")

In [6]:
all_results = {}

for short_name, model_name in MODEL_CONFIGS.items():
    start_bs = START_BATCH_SIZE.get(short_name, 32)

    results = evaluate_model_safely(
        short_name=short_name,
        model_name=model_name,
        root_path=GEODE_TEST_ROOT,
        target_to_imagenet_idx=GEODE_TO_IMAGENET_IDX,
        region_index=2,
        start_batch_size=start_bs,
    )

    all_results[short_name] = results

    with open("imagenet_pretrained_geode_results.pkl", "wb") as f:
        pickle.dump(all_results, f)

    clear_gpu()


Evaluating: convnextv2_base
timm model: convnextv2_base.fcmae_ft_in22k_in1k
batch size: 64

Device: cuda


model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

Loaded timm pretrained model
Model: convnextv2_base.fcmae_ft_in22k_in1k
Dataset classes: 40
Mapped classes: 34
Unmapped / skipped classes: 6
Unmapped: ['hairbrush_comb', 'spices', 'stall', 'toothbrush', 'toothpaste_toothpowder', 'tree']


  0%|          | 0/251 [00:00<?, ?it/s]

ImageNet-pretrained evaluation
Model: convnextv2_base.fcmae_ft_in22k_in1k

Accuracy:           63.37%
Evaluated images:   13517
Skipped images:     2538

Region bias summary

Best region:                     Americas
Worst region:                    WestAsia
Region gap:                      4.21%
Region std:                      1.71%

Object-balanced best region:     Americas
Object-balanced worst region:    WestAsia
Object-balanced region gap:      4.83%
Object-balanced region std:      1.69%

Evaluating: swinv2_base
timm model: swinv2_base_window12to24_192to384.ms_in22k_ft_in1k
batch size: 4

Device: cuda


...

ImageNet-pretrained evaluation
Model: deit3_base_patch16_224.fb_in22k_ft_in1k

Accuracy:           62.66%
Evaluated images:   13517
Skipped images:     2538

Region bias summary

Best region:                     Europe
Worst region:                    WestAsia
Region gap:                      2.73%
Region std:                      1.15%

Object-balanced best region:     Americas
Object-balanced worst region:    WestAsia
Object-balanced region gap:      3.94%
Object-balanced region std:      1.39%

Evaluating: dinov2_vitb14
timm model: vit_base_patch14_dinov2.lvd142m
batch size: 32

Device: cuda


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loaded timm pretrained model
Model: vit_base_patch14_dinov2.lvd142m
Dataset classes: 40
Mapped classes: 34
Unmapped / skipped classes: 6
Unmapped: ['hairbrush_comb', 'spices', 'stall', 'toothbrush', 'toothpaste_toothpowder', 'tree']


  0%|          | 0/502 [00:00<?, ?it/s]

ImageNet-pretrained evaluation
Model: vit_base_patch14_dinov2.lvd142m

Accuracy:           0.56%
Evaluated images:   13517
Skipped images:     2538

Region bias summary

Best region:                     SouthEastAsia
Worst region:                    EastAsia
Region gap:                      0.32%
Region std:                      0.14%

Object-balanced best region:     SouthEastAsia
Object-balanced worst region:    EastAsia
Object-balanced region gap:      0.29%
Object-balanced region std:      0.11%
